# Can we predict a warranty claim within a year?
Follows `protocol.yaml`: fixed holdout (`holdout.csv`), 5-fold stratified CV on training rows only, model chosen by mean CV ROC AUC, holdout scored once.

In [ ]:
import json, hashlib
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.base import clone

sha = lambda p: hashlib.sha256(open(p,'rb').read()).hexdigest()
assert sha('data.csv') == '25552b73fd810f7dfa81a14849736291d84f91da2f8b5a58bcfaf0cc018962f4'
assert sha('holdout.csv') == '083821b17b5c7fdf81de46633ca08a7d601bd8266b269faa3911938d26bf66bf'
df = pd.read_csv('data.csv')
hold_idx = np.sort(pd.read_csv('holdout.csv')['row'].values)  # 0-based positions in data.csv
assert len(df)==7000 and len(hold_idx)==1400 and len(set(hold_idx))==1400
train_idx = np.setdiff1d(np.arange(len(df)), hold_idx)
print(len(train_idx), len(hold_idx), df.claim_within_year.mean())

## Features
`device_serial` is excluded (identifier, per protocol). `channel`, `device_age_years`, `price_eur` and `daily_hours` are all recorded at sale/registration (see `columns.csv`), so known at prediction time. The target is the only post-sale column.

In [ ]:
TARGET='claim_within_year'
CAT=['channel']; NUM=['device_age_years','price_eur','daily_hours']
FEATURES=CAT+NUM
X, y = df[FEATURES], df[TARGET]
Xtr, ytr = X.iloc[train_idx], y.iloc[train_idx]
Xho, yho = X.iloc[hold_idx], y.iloc[hold_idx]
# EDA on training rows only
print(df.iloc[train_idx].groupby('channel')[TARGET].mean())
print(df.iloc[train_idx][NUM+[TARGET]].corr()[TARGET])

## Candidates (all named before any scoring)
Preprocessing lives inside each pipeline, so it is fit within each CV fold.

In [ ]:
def prep(scale):
    return ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore'),CAT),
                              ('n',StandardScaler() if scale else 'passthrough',NUM)])
candidates = {
 'baseline_prior': make_pipeline(prep(False), DummyClassifier(strategy='prior')),
 'logreg_C1': make_pipeline(prep(True), LogisticRegression(C=1.0, max_iter=2000)),
 'logreg_C0.1': make_pipeline(prep(True), LogisticRegression(C=0.1, max_iter=2000)),
 'random_forest': make_pipeline(prep(False), RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=0, n_jobs=-1)),
 'hist_gb': make_pipeline(prep(False), HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0)),
}
print(list(candidates))

## Cross-validation on training rows

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
folds = [(tr, va) for tr, va in skf.split(Xtr, ytr)]
cv_scores = {}
for name, model in candidates.items():
    s=[]
    for tr, va in folds:
        m = clone(model).fit(Xtr.iloc[tr], ytr.iloc[tr])
        s.append(float(roc_auc_score(ytr.iloc[va], m.predict_proba(Xtr.iloc[va])[:,1])))
    cv_scores[name]=s
cv = pd.DataFrame({k:{'mean':np.mean(v),'std':np.std(v)} for k,v in cv_scores.items()}).T
print(cv.sort_values('mean', ascending=False))

## Choice and single holdout score
Highest mean CV ROC AUC wins; the holdout is touched only after this.

In [ ]:
chosen = max(cv_scores, key=lambda k: np.mean(cv_scores[k]))
final = clone(candidates[chosen]).fit(Xtr, ytr)
holdout_auc = float(roc_auc_score(yho, final.predict_proba(Xho)[:,1]))
print(chosen, 'CV', round(np.mean(cv_scores[chosen]),4), 'holdout ROC AUC', round(holdout_auc,4))

In [ ]:
evidence = {
 'features': FEATURES,
 'candidates': list(candidates),
 'cv_folds': [[[int(i) for i in train_idx[tr]], [int(i) for i in train_idx[va]]] for tr,va in folds],  # (train, validation) rows per fold
 'cv_scores': cv_scores,
 'chosen': chosen,
 'holdout_rows': [int(i) for i in hold_idx],
 'holdout_scores': {chosen: holdout_auc},
}
json.dump(evidence, open('evidence.json','w'))

## Conclusion
See the printed holdout ROC AUC above: it is the headline estimate of how well claims within a year can be predicted from channel, device age, price and daily use.